In [ ]:
!pip install mapclassify
!pip install rasterio

In [ ]:
import geopandas
import pandas as pd
import folium
import matplotlib.pyplot as plt
import mapclassify
import shapely
from shapely.geometry import Point
import rasterio
import matplotlib
from matplotlib import pyplot
from rasterio.plot import show
from rasterio.plot import show_hist

In [ ]:
# Import a shapefile
buildings = geopandas.read_file("/content/buildings.geojson")
buildings = buildings.to_crs(epsg=4326)  # Convert to WGS84
newgdf = buildings.to_file("test.geojson", driver='GeoJSON')
buildings.plot(markersize=0.5)

In [ ]:
#Filter data
select_buildings = buildings[buildings['place_type'] == 'university building']
select_buildings.plot(markersize=0.5)

In [ ]:
#Challenge: Bring in a new layer, Texas counties and subset Travis county
# %%
texas_county = geopandas.read_file("/content/texas_county.geojson")
texas_county = texas_county.to_crs(epsg=4326)  # Convert to WGS84
texas_county.plot()
travis_boundary = texas_county[texas_county['name'] == 'Travis']
travis_boundary.plot()

In [ ]:
#Perform an intersection to select only buildings in Travis county
# %%
travis_buildings = buildings.overlay(travis_boundary, how='intersection')
travis_buildings.plot(markersize=1)


In [ ]:
#Plot multiple layers
# %%
base = travis_boundary.plot(color='white', edgecolor='black')
travis_buildings.plot(ax=base, marker='o', color='red', markersize=2);


In [ ]:
#Perform some more advanced operations
# %%
pop = geopandas.read_file("/content/travis_pop.geojson")
pop = pop.to_crs(epsg=32139)  # Convert to Texas Central UTM
pop.rename(columns={'VALUE0': 'population'}, inplace=True)
pop['population'] = pop['population'].astype(float)
pop.plot(column='population', cmap='YlOrRd', legend=True, legend_kwds={"orientation": "horizontal", "pad": 0.1})


In [ ]:
# Define a function to calculate population density
# %%
def calculate_population_density(gdf: geopandas.GeoDataFrame) -> geopandas.GeoDataFrame:
    gdf_copy = gdf.copy()
    # Calculate area in sq km for the entire column
    area_sq_km = gdf_copy.geometry.area / 1_000_000
    # Calculate population density using vectorized operations
    gdf_copy['pop_density'] = gdf_copy['population'] / area_sq_km
    return gdf_copy

In [ ]:
# Calculate population density and plot the map
pop_with_density = calculate_population_density(pop)
pop_with_density['pop_density'] = pop_with_density['pop_density'].astype(float)
pop_with_density.plot(column='pop_density', cmap='YlOrRd', legend=True, legend_kwds={"orientation": "horizontal", "pad": 0.1})
pop_with_density.plot(column='pop_density', scheme='natural_breaks')
newgdf = pop_with_density.to_file("travis_pop_with_density.geojson", driver='GeoJSON')

In [ ]:
# Import a raster file
sm = rasterio.open("/content/august_25_median_soil_moisture_texas.tif")
show(sm, cmap='pink')


In [ ]:
# Show histogram
show_hist(sm, bins=50)
show(sm, cmap='pink', vmin=0, vmax=0.3, title="Raster Data Visualization")